# Multivariate Gradient Descent Linear Regression - Raw Implementation

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

In [2]:
raw_data = pd.read_csv("house_prices_grad_desc.csv")

In [3]:
raw_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
features = ['LotFrontage', 'LotArea', 'TotalBsmtSF', 'GarageArea']
target = 'SalePrice'

In [5]:
data = raw_data[features + [target]].copy()

In [6]:
data.head()

,LotFrontage,LotArea,TotalBsmtSF,GarageArea,SalePrice
0,65.0,8450,856,548,208500
1,80.0,9600,1262,460,181500
2,68.0,11250,920,608,223500
3,60.0,9550,756,642,140000
4,84.0,14260,1145,836,250000


In [7]:
for feature in features:
    data[feature] = data[feature].fillna(data[feature].mean())

In [8]:
X = data[features].values
Y = data[target].values

In [9]:
X

array([[   65.,  8450.,   856.,   548.],
       [   80.,  9600.,  1262.,   460.],
       [   68., 11250.,   920.,   608.],
       ...,
       [   66.,  9042.,  1152.,   252.],
       [   68.,  9717.,  1078.,   240.],
       [   75.,  9937.,  1256.,   276.]], shape=(1460, 4))

In [10]:
Y

array([208500, 181500, 223500, ..., 266500, 142125, 147500], shape=(1460,))

In [11]:
# Feature scaling
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_std[X_std == 0] = 1e-8
X_scaled = (X - X_mean) / X_std

In [12]:
X_scaled

array([[-0.22937175, -0.20714171, -0.45930254,  0.35100032],
       [ 0.4519361 , -0.09188637,  0.46646492, -0.06073101],
       [-0.09311018,  0.07347998, -0.31336875,  0.63172623],
       ...,
       [-0.18395123, -0.14781027,  0.21564122, -1.03391416],
       [-0.09311018, -0.08016039,  0.04690528, -1.09005935],
       [ 0.22483348, -0.05811155,  0.45278362, -0.9216238 ]],
      shape=(1460, 4))

In [13]:
# Weight initialization
# w[0] = w_0, w[1] = w_1
w = np.ones(len(X[0]) + 1)

## Prediction / Hypothesis
$h_w(x) = w_0x_0 + w_1x_1 + \ldots + w_nx_n$

In [14]:
def predict(x_row, w):
    bias = w[0]
    prediction = 0.0

    for i in range(len(x_row)):
        prediction += w[i + 1] * x_row[i]

    return prediction + bias

## Cost Function
$J(w_0, w_1, \dots, w_n) = \frac{1}{2m}\sum_{i = 1}^m\{h_w(x_i) - Y_i\}^2 $

In [15]:
def cost_function(X, Y, w):
    current_cost = 0.0
    
    # number of data row
    m = len(Y)
    
    for i in range(m):
        current_cost += (predict(X[i], w) - Y[i])**2

    return current_cost / (2 * m)

## Update Function
$w_j = w_j - \alpha \frac{d}{dw_j}J(w_0, w_1, \dots, w_n)$

$w_j = w_j - \alpha \frac{1}{m}\sum_{i = 1}^m\{h_w(x_i) - Y_i\} x_{ij} $

In [16]:
def update_weights(X, Y, w, learning_rate):
    # Number of data
    m = len(Y)

    # Number of feature
    n = len(X[0])

    w_derivs = [0.0] * (n + 1)

    for i in range(m):
        error = predict(X[i], w) - Y[i] #hw(xi) - Yi

        w_derivs[0] += error 

        for j in range(n):
            w_derivs[j + 1] += error * X[i][j]

    for j in range(n+1):
        w[j] -= learning_rate * (w_derivs[j] / m)

    return w

## Main Training Loop

In [17]:
learning_rate = 0.1
epochs = 9000
tolerance = 1.0

print("Tarining Starts...")


prev_cost = cost_function(X_scaled, Y, w)

for epoch in range(1, epochs + 1):
    w = update_weights(X_scaled, Y, w, learning_rate)

    current_cost = cost_function(X_scaled, Y, w)
    
    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch}: Cost= {current_cost:,.5f}")

    cost_decrease = prev_cost - current_cost

    if tolerance > cost_decrease:
        print(f"\n[Convergence Reached] Stopped early at Epoch {epoch}!")
        print(f"The cost only decreased by {cost_decrease:.5f} (which is less than our tolerance of {tolerance})")
        print(f"Final Cost: {current_cost:,.5f}")
        break

    prev_cost = current_cost
    
print("Training Complete...")
print("-" * 50)
print(f"Final Bias (w_0)        : {w[0]:.2f}")

for j in range(len(X[0])):
    print(f"Final Weight {features[j]} : {w[j+1]:.5f}")

Tarining Starts...
Epoch 1: Cost= 15,869,422,362.54227
Epoch 50: Cost= 1,504,431,017.62006
Epoch 100: Cost= 1,503,897,956.87192

[Convergence Reached] Stopped early at Epoch 120!
The cost only decreased by 0.99526 (which is less than our tolerance of 1.0)
Final Cost: 1,503,897,853.74677
Training Complete...
--------------------------------------------------
Final Bias (w_0)        : 180920.61
Final Weight LotFrontage : 3181.16645
Final Weight LotArea : 6241.60106
Final Weight TotalBsmtSF : 30004.27890
Final Weight GarageArea : 32751.32588


In [18]:
test_data = np.array([80, 9500, 1500, 700])
test_data_scaled = (test_data - X_mean) / X_std

manual_prediction = predict(test_data_scaled, w)

print(f"Raw Model Prediction: ${manual_prediction:,.2f}")

Raw Model Prediction: $246,788.79


# Built-in Gradient Descent Linear Regression Multivariate

In [19]:
from sklearn.linear_model import SGDRegressor

In [20]:
# Initialize the built-in Gradient Descent Model
# max_iter = epochs
# eta0 = initial learning rate
sgd_model = SGDRegressor(max_iter=1000, eta0=0.1, random_state=42)

In [21]:
# Train the model using already scaled features and target!
sgd_model.fit(X_scaled, Y)

,"loss loss: str, default='squared_error'The loss function to be used. The possible values are 'squared_error','huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'The 'squared_error' refers to the ordinary least squares fit.'huber' modifies 'squared_error' to focus less on getting outlierscorrect by switching from squared to linear loss past a distance ofepsilon. 'epsilon_insensitive' ignores errors less than epsilon and islinear past that; this is the loss function used in SVR.'squared_epsilon_insensitive' is the same but becomes squared loss pasta tolerance of epsilon.More details about the losses formulas can be found in the:ref:`User Guide `.",'squared_error'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42


In [22]:
# Scikit-Learn stores the bias in .intercept_ and the weights in .coef_
b_builtin = sgd_model.intercept_[0]
w_builtin = sgd_model.coef_

In [23]:
print("Built-in Training Complete!")
print("-" * 50)
print(f"Final Bias (w_0)        : {b_builtin:.2f}")

for j in range(len(X[0])):
    print(f"Final Weight {features[j]} : {w_builtin[j]:.2f}")

Built-in Training Complete!
--------------------------------------------------
Final Bias (w_0)        : 176494.44
Final Weight LotFrontage : 2180.66
Final Weight LotArea : 57564.75
Final Weight TotalBsmtSF : 24930.98
Final Weight GarageArea : 33785.82


In [24]:
# 1. Take the test_data_scaled you already calculated manually
test_data_scaled_2d = test_data_scaled.reshape(1, -1)

# 2. Feed it directly into the built-in model
sklearn_prediction = sgd_model.predict(test_data_scaled_2d)[0]

print(f"Scikit-Learn Prediction: ${sklearn_prediction:,.2f}")

Scikit-Learn Prediction: $232,659.25
